In [1]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor, spearman_corr, top_k_recall
from surrogate_model.optuna import (
    RECALL_TOP_1_PERCENT,
    RECALL_TOP_5_PERCENT,
    make_objective,
)

In [2]:
FEATURES = "data/sampled.parquet"
LABELS = "data/1L83.1L83:p2rank:3.output.parquet"

RANDOM_SEED = 1000

features = pl.read_parquet(FEATURES)
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

df = features.join(labels, on="catalog_id", how="inner")

In [3]:
import logging

from e3fp.pipeline import fprints_from_mol
from rdkit import Chem


def generate_e3fp(sdf_str: str, bits: int = 1024):
    logging.basicConfig(level=logging.WARNING)
    logging.getLogger().setLevel(logging.WARNING)
    for logger_name in logging.root.manager.loggerDict:
        logging.getLogger(logger_name).setLevel(logging.WARNING)

    zero_vector = [0] * bits

    if not sdf_str:
        return zero_vector

    try:
        mol = Chem.MolFromMolBlock(sdf_str)
        if mol is None:
            return zero_vector

        if not mol.HasProp("_Name") or not mol.GetProp("_Name"):
            mol.SetProp("_Name", "molecule")

        fps = fprints_from_mol(mol, fprint_params={"bits": bits})

        if not fps:
            return zero_vector

        return fps[0].to_vector(sparse=False).astype(int).tolist()
    except Exception:
        return zero_vector

In [4]:
sdf_list = df["conformer_sdf"].to_list()

In [5]:
import concurrent.futures

from tqdm.auto import tqdm

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(
        tqdm(
            executor.map(generate_e3fp, sdf_list, chunksize=50),
            total=len(sdf_list),
            desc="Generating fingerprints",
        )
    )

Generating fingerprints:   0%|          | 0/107908 [00:00<?, ?it/s]

In [6]:
df = df.with_columns(pl.Series("e3fp", results, dtype=pl.List(pl.Int64)))

In [7]:
ALL_COLUMN_NAMES = [
    "catalog_id",
    "affinity_kcal_mol",
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
    "morgan_fingerprint",
    "e3fp",
]

df = df.select(ALL_COLUMN_NAMES)

In [8]:
FEATURE_NAMES = [
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
]

LABEL_NAME = "affinity_kcal_mol"

x_scalars = df.select(FEATURE_NAMES).to_numpy()

x_morgan = np.array(df["morgan_fingerprint"].to_list())
x_e3fp = np.array(df["e3fp"].to_list())

# x = np.hstack([x_scalars, x_morgan])
# x = np.hstack([x_scalars, x_e3fp])

x = np.hstack([x_scalars, x_morgan, x_e3fp])

y = df[LABEL_NAME].to_numpy()

In [27]:
SAMPLE_SIZE = 10000

x = x[:SAMPLE_SIZE]
y = y[:SAMPLE_SIZE]

# PRIMARY_METRIC = RECALL_TOP_1_PERCENT
PRIMARY_METRIC = RECALL_TOP_5_PERCENT
NUM_TRIALS = 20

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_SEED
)

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(
    make_objective(
        X_train,
        y_train,
        5,
        primary_metric=PRIMARY_METRIC,
        random_seed=RANDOM_SEED,
    ),
    n_trials=NUM_TRIALS,
    show_progress_bar=True,
)

[I 2026-09-08 14:35:40,191] A new study created in memory with name: no-name-8bea1156-a397-47df-b2cd-95cd171c3d89


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-09-08 14:35:41,592] Trial 0 finished with value: 0.125 and parameters: {'num_leaves': 173, 'max_depth': 4, 'learning_rate': 0.22592582730543753, 'n_estimators': 1016, 'min_child_samples': 88, 'subsample': 0.60616634046136, 'colsample_bytree': 0.5203548123845445, 'reg_alpha': 3.7562124870680294e-05, 'reg_lambda': 1.2536888868437446e-06}. Best is trial 0 with value: 0.125.
[I 2026-09-08 14:35:43,557] Trial 1 finished with value: 0.09 and parameters: {'num_leaves': 218, 'max_depth': 5, 'learning_rate': 0.06905371732106186, 'n_estimators': 845, 'min_child_samples': 22, 'subsample': 0.87176970729607, 'colsample_bytree': 0.5347910404849773, 'reg_alpha': 0.9290409121444837, 'reg_lambda': 3.7480000930162687}. Best is trial 0 with value: 0.125.
[I 2026-09-08 14:35:50,369] Trial 2 finished with value: 0.10500000000000001 and parameters: {'num_leaves': 240, 'max_depth': 7, 'learning_rate': 0.001179752983913515, 'n_estimators': 1966, 'min_child_samples': 37, 'subsample': 0.8533435969444019

In [28]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params, deterministic=True, force_row_wise=True)
final_model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

y_pred = np.asarray(final_model.predict(X_test))

results = {
    "top_1_percent": top_k_recall(y_test, y_pred, 0.01),
    "top_5_percent": top_k_recall(y_test, y_pred, 0.05),
    "top_10_percent": top_k_recall(y_test, y_pred, 0.1),
    "spearman": spearman_corr(y_test, y_pred),
    "enrichment_factor_1_percent": enrichment_factor(y_test, y_pred, 0.01),
    "enrichment_factor_5_percent": enrichment_factor(y_test, y_pred, 0.05),
    "enrichment_factor_10_percent": enrichment_factor(y_test, y_pred, 0.1),
}

results

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[210]	valid_0's l2: 261.14


{'top_1_percent': 0.3,
 'top_5_percent': 0.26,
 'top_10_percent': 0.33,
 'spearman': 0.2938822002329909,
 'enrichment_factor_1_percent': 30.0,
 'enrichment_factor_5_percent': 5.2,
 'enrichment_factor_10_percent': 3.3}

| dataset size | train metric | trials | top_1_percent | top_5_percent | top_10_percent | spearman | enrichment 1 percent | enrichment 5 percent | enrichment 10 percent |
| - | - | - | - | - | - | - | - | - | - |
| 10000 | RECALL_TOP_1_PERCENT | 20 |  | | | | | |  |
| 10000 | RECALL_TOP_5_PERCENT | 20 | 0.3 | 0.26 | 0.33 | 0.294 | 30.0 | 5.2 | 3.3 |